<a href="https://colab.research.google.com/github/YuriArduino/Estudos_Artificial_Intelligence/blob/Lang_chain/Lang_chain_atualizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ## Roteiro de Estudos Otimizado: LangChain com Google Gemini

**Criado por:** Yuri Arduino

**Objetivo:** Este notebook é um guia atualizado e otimizado para estudos de LangChain, focado na construção de sistemas de RAG (Retrieval-Augmented Generation).

**Otimizações aplicadas:**
- **Organização:** Estrutura modular com explicações claras.
- **Pydantic v2:** Uso de `pydantic-settings` para gerenciamento de configurações.
- **Performance:** Otimização para GPU em modelos de embedding locais.
- **Boas Práticas:** Código limpo, comentado e seguindo as versões mais recentes das bibliotecas.

---

#1. Instalação de Dependências

 Instalação das bibliotecas essenciais:
 - langchain e langchain-google-genai: Para orquestração e integração com a API do Gemini.
 - pypdf e unstructured: Para carregar e extrair texto de documentos PDF.
 - faiss-cpu e chromadb: Vector stores para armazenar e buscar embeddings localmente.
 - sentence-transformers: Necessário para os modelos de embedding do Hugging Face.

In [ ]:
# Célula de Instalação ÚNICA E COMPLETA (ATUALIZADA)
!pip install -q --upgrade langchain langchain-core langchain-community langchain-google-genai \
pypdf sentence-transformers faiss-cpu chromadb "pydantic-settings>2.0.0" \
langchain-pinecone pinecone-client unstructured[pdf] duckdb \
pandas plotly umap-learn scikit-learn kaleido \
ragas datasets
print("✅ Todas as bibliotecas foram instaladas com sucesso!")

In [ ]:
import langchain
print(langchain.__version__)

##1.1. Configuração do Ambiente (Logging e API Keys) (Código)

In [ ]:
import os
import logging
from datetime import datetime
import pytz
from google.colab import userdata

# --- Configuração do Logging com Fuso Horário de Brasília ---
# Um bom sistema de logs é essencial para depurar e entender o que está acontecendo.
brasilia_tz = pytz.timezone("America/Sao_Paulo")

class TZFormatter(logging.Formatter):
    def formatTime(self, record, datefmt=None):
        dt = datetime.fromtimestamp(record.created, tz=brasilia_tz)
        return dt.strftime(datefmt or "%H:%M:%S")

handler = logging.StreamHandler()
handler.setFormatter(TZFormatter("%(asctime)s | %(levelname)-7s | %(message)s"))
logging.basicConfig(level=logging.INFO, handlers=[handler], force=True)
logger = logging.getLogger(__name__)

logger.info("Logging configurado com sucesso - Horário de Brasília (UTC-3).")

# --- Configuração da API Key do Google ---
# Carrega a chave da API do Gemini a partir dos "Secrets" do Google Colab.
# Esta é uma prática de segurança para não expor suas chaves no código.
try:
    api_key = userdata.get("GEMINI_API_KEY")
    os.environ["GOOGLE_API_KEY"] = api_key
    logger.info("GEMINI_API_KEY carregada com sucesso.")
except Exception as e:
    logger.error("Chave GEMINI_API_KEY não encontrada nos Secrets do Colab. Por favor, adicione-a.")
    raise e

##1.2. Carregador do Modelo LLM

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

def carregar_llm(model: str = "gemini-2.5-flash", temperature: float = 0):
    """Carrega e configura o modelo LLM do Google Gemini."""
    try:
        # Adiciona uma camada de conversão para mensagens de sistema, uma boa prática para alguns modelos
        llm = ChatGoogleGenerativeAI(
            model=model,
            temperature=temperature,
            convert_system_message_to_human=True
        )
        logger.info(f"✅ Modelo LLM '{model}' carregado com sucesso!")
        return llm
    except Exception as e:
        logger.error(f"❌ Erro ao carregar o modelo LLM: {e}")
        return None

def carregar_embedding_model(model: str = "gemini-embedding-001"):
    """Carrega e configura o modelo de embedding do Google."""
    try:
        embeddings = GoogleGenerativeAIEmbeddings(model=model)
        logger.info(f"✅ Modelo de Embedding '{model}' carregado com sucesso!")
        return embeddings
    except Exception as e:
        logger.error(f"❌ Erro ao carregar o modelo de embedding: {e}")
        return None

# Carrega os modelos principais que serão usados no notebook
llm_principal = carregar_llm()
embedding_principal = carregar_embedding_model()

# Verificação opcional para confirmar que os objetos foram criados
if llm_principal and embedding_principal:
    logger.info("LLM principal e modelo de embedding prontos para uso.")
else:
    logger.error("Falha ao inicializar um ou mais modelos. Verifique as mensagens de erro acima.")

# Seção 1: Fundamentos de RAG (Retrieval-Augmented Generation)

Nesta seção, vamos explorar a diferença entre uma abordagem de LLM tradicional e uma abordagem RAG.

**Cenário:** Queremos perguntar ao nosso LLM sobre a política de home office de uma empresa fictícia. O modelo, por padrão, não tem essa informação.

**1. Abordagem Tradicional (Sem RAG):** O LLM tentará responder com base em seu conhecimento geral, o que geralmente resulta em uma resposta genérica ou incorreta.

**2. Abordagem RAG:** Nós fornecemos ao LLM o documento exato com a política (o "contexto") e pedimos que ele baseie sua resposta *apenas* nesse contexto.

##Exemplo Prático - Abordagem Tradicional

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

pergunta = "Qual é a política de home office da nossa empresa?"

# Chain Tradicional
prompt_tradicional = ChatPromptTemplate.from_template("Responda a seguinte pergunta: {pergunta}")
chain_tradicional = prompt_tradicional | llm_principal

# Execução
logger.info("Executando a cadeia tradicional (sem RAG)...")
resposta_tradicional = chain_tradicional.invoke({"pergunta": pergunta})

print("\n" + "="*50)
print(f"Pergunta: {pergunta}")
print(f"Resposta (Sem RAG): {resposta_tradicional.content}")
print("="*50)
logger.warning("Observe que a resposta é genérica e inventada.")

##Exemplo Prático - Abordagem RAG

In [ ]:
import time
from pathlib import Path
from typing import List
from urllib.parse import unquote
import requests
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# --- 1. Lógica de Download (Robusta) ---
class DocumentDownloader:
    def __init__(self, persist_dir: str = "/content/data"):
        self.persist_dir = Path(persist_dir)
        self.persist_dir.mkdir(parents=True, exist_ok=True)

    def _get_filename_from_url(self, url: str) -> str:
        return unquote(url.split("/")[-1].split("?")[0])

    def _convert_github_url(self, url: str) -> str:
        if "github.com" in url and "/blob/" in url:
            logger.info("URL do GitHub detectada. Convertendo para formato raw...")
            return url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
        return url

    def download(self, url: str) -> Path:
        raw_url = self._convert_github_url(url)
        filename = self._get_filename_from_url(raw_url)
        filepath = self.persist_dir / filename

        if filepath.exists():
            logger.info(f"Arquivo '{filename}' já existe. Usando cache.")
            return filepath

        try:
            logger.info(f"Baixando '{filename}' de {raw_url}...")
            response = requests.get(raw_url, timeout=30)
            response.raise_for_status()
            filepath.write_bytes(response.content)
            logger.info(f"✅ Arquivo salvo em: {filepath}")
            return filepath
        except requests.RequestException as e:
            logger.error(f"❌ Falha no download de {raw_url}: {e}")
            return None

# --- 2. Execução do Pipeline RAG ---

# LISTA DE URLs CORRIGIDA (adicionando a branch 'Lang_chain')
urls = [
    "https://github.com/YuriArduino/Estudos_Artificial_Intelligence/blob/Dados/politica_home_office.pdf",
    "https://github.com/YuriArduino/Estudos_Artificial_Intelligence/blob/Dados/relatorio_vendas.pdf"
]

# Vamos usar apenas o primeiro PDF para este exemplo
pdf_url = urls[0]

downloader = DocumentDownloader()
pdf_path = downloader.download(pdf_url)

if pdf_path:
    logger.info(f"Carregando texto do PDF: {pdf_path}...")
    loader = PyPDFLoader(str(pdf_path))
    documento_pdf = loader.load()
    contexto_empresa = documento_pdf[0].page_content
    logger.info("✅ Contexto extraído do PDF com sucesso.")

    pergunta = "Qual é a política de home office da nossa empresa?"

    prompt_rag = ChatPromptTemplate.from_template(
        """
        Você é um assistente de RH. Responda a pergunta do usuário baseando-se estritamente no contexto fornecido.
        Se a informação não estiver no contexto, diga "Com base no documento, não tenho informações sobre isso."

        **Contexto:**
        {contexto}

        **Pergunta:**
        {pergunta}
        """
    )

    # O `llm_model` deve ter sido criado em uma célula anterior
    chain_rag = (
        {"contexto": lambda x: contexto_empresa, "pergunta": lambda x: x["pergunta"]}
        | prompt_rag
        | llm_principal
    )

    logger.info("Invocando a cadeia RAG...")
    resposta_rag = chain_rag.invoke({"pergunta": pergunta})

    print("\n" + "="*50)
    print(f"Pergunta: {pergunta}")
    print(f"Resposta (Com RAG): {resposta_rag.content}")
    print("="*50)
else:
    logger.error("Pipeline RAG não pôde ser executado pois o download do PDF falhou.")

# Seção 2: Embeddings e Vector Stores - O Cérebro da Memória do RAG

Para que um sistema RAG funcione com dezenas ou milhares de documentos, é impossível enviar todos eles como contexto para o LLM a cada pergunta. Precisamos de uma estratégia inteligente para encontrar, em milissegundos, os trechos de informação mais relevantes para a dúvida do usuário.

É aqui que a mágica acontece, através de dois componentes fundamentais: **Embeddings** e **Vector Stores**.

### O que são Embeddings? A Tradução da Semântica

Um embedding é a representação numérica do significado de um texto. Um modelo de embedding especializado lê um trecho de texto e o transforma em um **vetor** (uma longa lista de números).

A principal característica é que textos com significados parecidos, como *"qual a regra para tirar férias?"* e *"como solicito meus dias de descanso?"*, terão vetores matematicamente próximos no espaço vetorial. É como dar um endereço (coordenadas) para o significado de cada pedaço de texto.

### O que são Vector Stores? A Biblioteca Otimizada

Um Vector Store é um tipo de banco de dados construído especificamente para armazenar esses vetores e realizar buscas por similaridade em altíssima velocidade. Quando uma nova pergunta chega, nós a transformamos em um vetor e pedimos ao Vector Store para "encontrar os vetores mais próximos", que correspondem aos documentos mais relevantes para responder àquela pergunta.


### Nossas Ferramentas de Trabalho

Nesta seção, vamos colocar a mão na massa e comparar três das mais populares e versáteis soluções de Vector Stores, cada uma adequada para um cenário diferente:

1.  **FAISS:** Desenvolvido pelo Facebook AI, é extremamente rápido e opera **em memória**. Perfeito para prototipagem rápida e cenários onde os dados não são massivos.
2.  **ChromaDB:** Uma solução moderna e fácil de usar que **persiste os dados em disco**, facilitando o reuso sem a necessidade de reprocessar tudo. Oferece um poderoso sistema de filtragem por metadados.
3.  **Pinecone:** Um serviço gerenciado **na nuvem**, projetado para produção e grande escala. Oferece alta disponibilidade, escalabilidade e recursos avançados para aplicações robustas.

###Instalações e Preparação

In [ ]:
import os
import shutil
from langchain_core.documents import Document

# --- Componentes Comuns para a Seção de Vector Stores ---

# O modelo de embedding já foi carregado na Seção 0.3 como 'embedding_principal'. Vamos reutilizá-lo.
# Para garantir que o código seja robusto, vamos determinar a dimensão do vetor dinamicamente.
try:
    logger.info("Verificando a dimensão do modelo de embedding...")
    test_vector = embedding_principal.embed_query("texto de teste para obter a dimensão")
    EMBEDDING_DIMENSION = len(test_vector)
    logger.info(f"✅ Dimensão do vetor detectada dinamicamente: {EMBEDDING_DIMENSION}")
except Exception as e:
    logger.error(f"❌ Não foi possível determinar a dimensão do embedding. Usando valor padrão 768. Erro: {e}")
    EMBEDDING_DIMENSION = 768 # Fallback para um valor padrão conhecido

# Documentos de exemplo com metadados ricos que usaremos para testar FAISS, Chroma e Pinecone
documentos_empresa = [
    Document(
        page_content="Política de férias: Funcionários têm direito a 30 dias de férias após 12 meses. A solicitação deve ser feita com 30 dias de antecedência.",
        metadata={"tipo": "política", "departamento": "RH", "ano": 2024, "id_doc": "doc001"}
    ),
    Document(
        page_content="Processo de reembolso de despesas: Envie a nota fiscal pelo portal financeiro. O reembolso ocorre em até 5 dias úteis.",
        metadata={"tipo": "processo", "departamento": "Financeiro", "ano": 2023, "id_doc": "doc002"}
    ),
    Document(
        page_content="Guia de TI: Para configurar a VPN, acesse vpn.nossaempresa.com e siga as instruções para seu sistema operacional.",
        metadata={"tipo": "tutorial", "departamento": "TI", "ano": 2024, "id_doc": "doc003"}
    ),
    Document(
        page_content="Código de Ética e Conduta: Valorizamos o respeito, a integridade e a colaboração. Casos de assédio não serão tolerados.",
        metadata={"tipo": "política", "departamento": "RH", "ano": 2022, "id_doc": "doc004"}
    )
]

logger.info(f"✅ {len(documentos_empresa)} documentos de exemplo prontos para serem vetorizados.")

###Vector Store 1 - FAISS (Rápido e em Memória)

In [ ]:
from langchain_community.vectorstores import FAISS

logger.info("--- Testando Vector Store: FAISS ---")

# A LangChain abstrai a complexidade de criar o índice FAISS.
# `FAISS.from_documents` automaticamente:
# 1. Vetoriza cada documento usando o `embedding_principal`.
# 2. Cria um índice FAISS otimizado para busca.
# 3. Armazena os vetores e os documentos no índice.
faiss_db = FAISS.from_documents(documentos_empresa, embedding_principal)

pergunta = "Como peço minhas férias?"
resultados = faiss_db.similarity_search(pergunta, k=2)

print("\n" + "="*50)
print(f"🔍 Pergunta: '{pergunta}'")
print("📄 Documentos mais relevantes (FAISS):")
for doc in resultados:
    print(f"- Conteúdo: {doc.page_content}")
    print(f"  (Metadados: {doc.metadata})")
print("="*50)

###Vector Store 2 - ChromaDB (Persistente e com Filtros)

Diferente do FAISS, que opera em memória, o **ChromaDB** é uma solução que **persiste os dados em disco**. Isso significa que, uma vez criado o banco de dados vetorial, podemos recarregá-lo em sessões futuras sem precisar reprocessar todos os documentos.

Sua principal vantagem é um poderoso sistema de **filtragem por metadados**, permitindo buscas semânticas combinadas com filtros exatos (ex: buscar por "férias" apenas em documentos do "RH" do ano "2024").

In [ ]:
from langchain_community.vectorstores import Chroma

logger.info("\n--- Testando Vector Store: ChromaDB ---")

# Boa prática: Definir um diretório e garantir que ele esteja limpo para cada execução do notebook.
# Isso evita o uso de dados de uma execução anterior.
persist_directory = "./chroma_db_persist"
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)
logger.info(f"Diretório de persistência '{persist_directory}' limpo para um novo início.")

# Criando o ChromaDB com persistência.
# Note que estamos reutilizando a variável 'embedding_principal' definida na Seção 0.
chroma_db = Chroma.from_documents(
    documents=documentos_empresa,
    embedding=embedding_principal, # Reutilizamos o modelo de embedding já carregado.
    persist_directory=persist_directory,
)
logger.info(f"✅ ChromaDB criado e persistido. Contém {chroma_db._collection.count()} documentos.")


# --- Teste 1: Busca por similaridade simples ---
pergunta_simples = "Como peço minhas férias?"
resultados_simples = chroma_db.similarity_search(pergunta_simples, k=2)

print("\n" + "="*60)
print(f"🔍 PERGUNTA SIMPLES: '{pergunta_simples}'")
print("\n📄 Documentos mais relevantes (ChromaDB - Busca Simples):")
for doc in resultados_simples:
  print(f"- Conteúdo: {doc.page_content}")
  print(f"  (Metadados: {doc.metadata})")
print("="*60)


# --- Teste 2: Busca com filtro de metadados ---
pergunta_filtrada = "Quais são as regras da empresa para o RH?"

# O filtro permite combinar condições. Aqui, queremos documentos que sejam
# do departamento "RH" E do tipo "política".
filtro = {
    "$and": [
        {"departamento": {"$eq": "RH"}},
        {"tipo": {"$eq": "política"}}
    ]
}

resultados_filtrados = chroma_db.similarity_search(
    pergunta_filtrada,
    k=2, # Busca os 2 mais relevantes que atendem ao filtro
    filter=filtro
)

print("\n" + "="*60)
print(f"🔍 PERGUNTA COM FILTRO: '{pergunta_filtrada}' (Filtro: dept='RH' E tipo='política')")
print("\n📄 Documentos relevantes (ChromaDB - Busca com Filtro):")
for doc in resultados_filtrados:
  print(f"- Conteúdo: {doc.page_content}")
  print(f"  (Metadados Confirmados: Departamento='{doc.metadata['departamento']}', Tipo='{doc.metadata['tipo']}')")
print("="*60)

###Vector Store 3 - Pinecone (Nuvem e Escalável)

Pinecone é uma solução de nível profissional. Para usá-lo, precisamos configurar nossas chaves de API de forma segura. A abordagem a seguir, usando `userdata` do Colab e `Pydantic`, é a **melhor prática** para gerenciar configurações a partir de variáveis de ambiente. Isso garante que nosso código não prossiga se alguma configuração essencial (como a chave de API) estiver faltando ou for inválida.

###1: Configuração Segura com Pydantic v2

In [ ]:
from pydantic import Field, ValidationError, field_validator
from pydantic_settings import BaseSettings

# A chave PINECONE_API_KEY já foi carregada para o ambiente na Seção 0.2.
# Esta célula agora foca exclusivamente em validar e estruturar
#essas configurações usando Pydantic.

class PineconeSettings(BaseSettings):
    """
    Carrega e valida as configurações do Pinecone a partir de variáveis de ambiente.
    """
    pinecone_api_key: str = Field(..., env="PINECONE_API_KEY")
    index_name: str = Field("langchain-rag-aula", env="PINECONE_INDEX_NAME") # Nome padrão do índice
    cloud: str = Field("aws", env="PINECONE_CLOUD") # Provedor de nuvem padrão
    region: str = Field("us-east-1", env="PINECONE_REGION") # Região padrão

    @field_validator("pinecone_api_key")
    @classmethod
    def check_key_not_empty(cls, v: str) -> str:
        """Garante que a chave da API não seja uma string vazia."""
        if not v or not v.strip():
            raise ValueError("A variável de ambiente PINECONE_API_KEY não pode ser vazia.")
        return v

    # Pydantic v2 usa model_config para configurações da classe
    class Config:
        env_file = '.env'
        env_file_encoding = 'utf-8'
        extra = 'ignore'

try:
    settings = PineconeSettings()
    logger.info("✅ Configurações do Pinecone validadas com sucesso.")
    # Para fins de segurança, não logamos a chave completa
    logger.info(f"Índice: '{settings.index_name}', Nuvem: '{settings.cloud}', Região: '{settings.region}'")
except (ValidationError, ValueError) as e:
    logger.error(f"❌ Erro nas configurações do Pinecone. Verifique suas variáveis de ambiente nos Secrets do Colab.")
    logger.error(f"Detalhe do erro: {e}")
    # Interrompe a execução se as configurações forem inválidas
    raise

###2: Lógica de Criação/Conexão do Índice Pinecone

Para manter nosso código organizado e reutilizável, criamos uma função auxiliar que encapsula toda a lógica de gerenciamento do índice no Pinecone.

**Responsabilidades desta função:**
1.  Verificar se o índice já existe.
2.  Se existir, conferir se a dimensão do vetor é compatível com nosso modelo de embedding atual.
3.  Se a dimensão for incompatível, apagar o índice antigo para evitar erros.
4.  Criar um novo índice com a dimensão correta.
5.  Aguardar o índice ficar pronto para uso.
6.  Popular o índice com nossos documentos.
7.  Retornar um objeto `Pinecone` da LangChain, pronto para ser usado.

In [ ]:
# Célula de Definição da Função
import time
from pinecone import Pinecone as PineconeClient, ServerlessSpec
from langchain_pinecone import Pinecone
from langchain_core.embeddings import Embeddings

def get_or_create_pinecone_db(
    pinecone_client: PineconeClient,
    index_name: str,
    embedding_model: Embeddings,
    documents: list,
    spec: ServerlessSpec
) -> Pinecone:
    """
    Verifica um índice Pinecone, cria ou recria se necessário,
    e retorna um objeto LangChain Pinecone.
    """
    logger.info(f"--- Gerenciando Índice Pinecone: '{index_name}' ---")

    try:
        # Usa a variável global EMBEDDING_DIMENSION que já calculamos dinamicamente.
        current_dimension = EMBEDDING_DIMENSION
        logger.info(f"Dimensão requerida pelo modelo de embedding: {current_dimension}")
    except NameError:
        logger.error("A variável global EMBEDDING_DIMENSION não foi definida. Execute a célula de preparação.")
        raise

    if index_name in pinecone_client.list_indexes().names():
        logger.info(f"Índice '{index_name}' encontrado. Verificando metadados...")
        index_info = pinecone_client.describe_index(index_name)

        if index_info.dimension == current_dimension:
            logger.info("Dimensão compatível. Conectando ao índice existente...")
            return Pinecone.from_existing_index(index_name, embedding_model)
        else:
            logger.warning(f"INCOMPATIBILIDADE DE DIMENSÃO! Índice tem {index_info.dimension}, mas o modelo gera {current_dimension}.")
            logger.warning(f"Excluindo o índice '{index_name}' para recriá-lo...")
            pinecone_client.delete_index(index_name)
            # Aguarda a exclusão ser confirmada pela API
            while index_name in pinecone_client.list_indexes().names():
                logger.info("Aguardando a exclusão do índice antigo...")
                time.sleep(2)
            logger.info("Índice antigo excluído com sucesso.")

    logger.info(f"Criando novo índice '{index_name}' com dimensão {current_dimension}...")
    pinecone_client.create_index(
        name=index_name, dimension=current_dimension, metric="cosine", spec=spec
    )
    # Aguarda o índice ficar pronto
    while not pinecone_client.describe_index(index_name).status['ready']:
        logger.info("Aguardando o índice ficar pronto...")
        time.sleep(2)

    logger.info(f"Índice '{index_name}' criado. Populando com {len(documents)} documentos...")
    pinecone_db = Pinecone.from_documents(
        documents, embedding_model, index_name=index_name
    )
    logger.info("✅ Documentos inseridos com sucesso no Pinecone.")
    return pinecone_db

### 2.6. Execução do Pipeline e Teste de Buscas no Pinecone

Agora, com nossa função auxiliar definida, vamos executar o pipeline. Instanciamos o cliente do Pinecone com nossas configurações validadas e chamamos a função para obter nosso Vector Store.

In [ ]:
# Célula de Execução e Teste

# O objeto 'settings' foi criado e validado na célula anterior.
pinecone_db = None # Inicializa a variável para garantir que ela exista
try:
    # --- 1. Execução Principal ---
    pinecone_client = PineconeClient(api_key=settings.pinecone_api_key)
    spec = ServerlessSpec(cloud=settings.cloud, region=settings.region)

    pinecone_db = get_or_create_pinecone_db(
        pinecone_client=pinecone_client,
        index_name=settings.index_name,
        embedding_model=embedding_principal, # Reutiliza o modelo de embedding global
        documents=documentos_empresa,
        spec=spec
    )

except Exception as e:
    logger.error(f"❌ Falha ao inicializar ou criar o índice no Pinecone: {e}")
    logger.error("Verifique se sua API Key do Pinecone está correta e se seu projeto está ativo.")

# --- 2. Uso do Vector Store ---
if pinecone_db:
    # --- Teste 1: Busca por Similaridade Simples ---
    pergunta_simples = "Como configuro a VPN?"
    resultados_simples = pinecone_db.similarity_search(pergunta_simples, k=1)

    print("\n" + "="*60)
    print(f"🔍 PERGUNTA SIMPLES: '{pergunta_simples}'")
    print("\n📄 Documentos mais relevantes (Pinecone - Busca Simples):")
    for doc in resultados_simples:
        print(f"- Conteúdo: {doc.page_content}")
        print(f"  (Metadados: {doc.metadata})")
    print("="*60)

    # --- Teste 2: Busca com Filtro ---
    # A sintaxe de filtro do Pinecone para igualdade simples é um dicionário direto.
    pergunta_filtrada = "informações sobre regras"
    resultados_filtrados = pinecone_db.similarity_search(
        pergunta_filtrada,
        k=2,
        filter={"tipo": "política"}
    )

    print("\n" + "="*60)
    print(f"🔍 PERGUNTA COM FILTRO: '{pergunta_filtrada}' (Filtro: tipo='política')")
    print("\n📄 Documentos relevantes (Pinecone - Busca com Filtro):")
    for doc in resultados_filtrados:
        print(f"- Conteúdo: {doc.page_content}")
        print(f"  (Metadados Confirmados: Tipo='{doc.metadata['tipo']}')")
    print("="*60)

else:
    logger.error("A instância do Vector Store do Pinecone não pôde ser criada. As buscas não serão executadas.")

---

## Para saber mais **HNSW (Hierarchical Navigable Small World)**

## **1. Estrutura do HNSW**

O HNSW é um índice baseado em **grafos** que organiza vetores de forma hierárquica.

* Cada nó do grafo representa um item do conjunto de dados.
* Cada nó mantém ligações com seus vizinhos mais próximos.

Essa estrutura possibilita **saltos estratégicos** durante a busca, acelerando a recuperação dos itens mais semelhantes — ainda que não se atinja a mesma precisão de uma busca exaustiva.

---

## **2. Papel do Número de Vizinhos**

Um dos parâmetros essenciais na configuração do HNSW é o número de vizinhos conectados a cada nó (*geralmente definido como 32*). Esse parâmetro afeta diretamente:

* **Qualidade da busca**

  * Um número maior de vizinhos tende a aumentar o *recall* (probabilidade de encontrar itens realmente próximos ao vetor de consulta).
  * Especialmente útil em dados com **alta variabilidade**.

* **Desempenho e custo computacional**

  * Mais conexões aumentam o uso de memória e o tempo de construção do índice.
  * Valores muito altos podem gerar lentidão em ambientes com **recursos limitados**.

---

## **3. Considerações na Escolha do Valor**

A definição do número de vizinhos é um **trade-off entre velocidade e acurácia**:

* Projetos que priorizam **precisão** e dispõem de **recursos robustos** → usar valores mais altos.
* Cenários com **grandes volumes de dados** ou **protótipos iniciais** → valores menores oferecem respostas mais rápidas, embora com leve perda de precisão.

A escolha ideal exige **testes práticos** e aferição de métricas de similaridade, sempre alinhada aos objetivos do projeto.

---

## **4. Exemplo Prático em Código**

```python
import faiss

# Dimensão dos vetores
d = 768

# Número de vizinhos configurados para o índice HNSW
M = 32

# Criação do índice HNSW
index = faiss.IndexHNSWFlat(d, M)
```

Nesse exemplo:

* Vetores de dimensão **768**.
* Cada nó mantém **32 conexões**.
* Essa configuração serve como **ponto de partida** e pode ser ajustada conforme a avaliação de desempenho e a natureza dos dados.

---

## **5. Conclusão**

A experimentação com diferentes valores de vizinhos é crucial para encontrar o **equilíbrio ideal**:

* **Buscas rápidas**.
* **Resultados relevantes**.
* **Uso eficiente de recursos**.

---



# Seção 3: Embeddings de Alta Performance - Comparando Velocidade e Qualidade

A escolha do modelo de embedding impacta diretamente dois eixos críticos de um sistema RAG:

1.  **Performance (Velocidade):** O quão rápido conseguimos transformar um grande volume de documentos em vetores? Isso é crucial para a etapa de ingestão e indexação de dados.
2.  **Qualidade (Semântica):** O quão bem o modelo entende a nuance e a intenção por trás de uma pergunta para encontrar os documentos mais relevantes?

Nesta seção, faremos um benchmark completo, comparando quatro modelos populares em ambos os eixos, com um foco especial em otimização de hardware (CPU vs. GPU).

##Preparação do Ambiente e do Benchmark

Primeiro, preparamos os componentes para nossa análise: definimos os modelos que serão testados, criamos um conjunto de textos de exemplo e verificamos se uma GPU está disponível para acelerar os modelos locais.

In [ ]:
import time
import torch
import pandas as pd
import plotly.express as px
from langchain_community.embeddings import HuggingFaceEmbeddings

# --- 1. Verificação de Hardware ---
# Essencial para garantir que os modelos Hugging Face usem a GPU, se disponível.
if torch.cuda.is_available():
    DEVICE = "cuda"
    logger.info(f"✅ GPU detectada: {torch.cuda.get_device_name(0)}. Modelos locais usarão {DEVICE}.")
else:
    DEVICE = "cpu"
    logger.warning("⚠️ GPU não detectada. Modelos locais rodarão na CPU, o que será consideravelmente mais lento.")

# --- 2. Dados de Teste ---
# Usaremos o mesmo conjunto de frases para todos os modelos para uma comparação justa.
textos_teste_benchmark = [
    "Qual é a política de férias da nossa empresa?",
    "Preciso de um relatório de despesas de viagem.",
    "Como configuro o acesso à rede privada virtual (VPN)?",
    "Onde encontro o código de conduta da organização?",
    "Quero entender o processo de avaliação de performance."
]

# --- 3. Definição dos Modelos para o Benchmark ---
# Reutilizamos 'embedding_principal' (Gemini) e inicializamos os outros modelos,
# otimizando-os para usar a GPU (DEVICE).
modelos_para_testar = {
    "Gemini (API)": embedding_principal, # Reutilizando nosso modelo principal
    "Multilingual-E5 (Local)": HuggingFaceEmbeddings(
        model_name="intfloat/multilingual-e5-large",
        model_kwargs={'device': DEVICE}
    ),
    "MiniLM (Local)": HuggingFaceEmbeddings(
        model_name="all-MiniLM-L6-v2",
        model_kwargs={'device': DEVICE}
    ),
    "BGE-Large (Local)": HuggingFaceEmbeddings(
        model_name="BAAI/bge-large-en-v1.5",
        model_kwargs={'device': DEVICE},
        encode_kwargs={'normalize_embeddings': True} # BGE recomenda normalização
    )
}

logger.info(f"{len(modelos_para_testar)} modelos foram configurados para o benchmark.")

##Benchmark de Performance

Agora, medimos o tempo que cada modelo leva para vetorizar nosso conjunto de textos. Isso nos dá uma ideia clara do trade-off entre o tamanho do modelo e a velocidade de processamento.

- **Modelos de API (Gemini):** A velocidade depende da latência da rede e da carga nos servidores do Google.
- **Modelos Locais (Hugging Face):** A velocidade depende diretamente do nosso hardware (GPU vs. CPU).

In [ ]:
# --- Execução do Benchmark de Performance ---
logger.info(f"Iniciando benchmark de performance com {len(textos_teste_benchmark)} documentos...")
resultados_performance = []
embeddings_gerados = {} # Dicionário para armazenar os vetores para a análise de qualidade

for nome, modelo in modelos_para_testar.items():
    logger.info(f"Processando com o modelo: {nome}...")
    start_time = time.time()

    # Gera os embeddings para o lote de textos
    vetores = modelo.embed_documents(textos_teste_benchmark)

    end_time = time.time()
    tempo_total = end_time - start_time

    # Armazena os vetores para uso posterior
    embeddings_gerados[nome] = vetores

    # Coleta as métricas de performance
    dimensao_vetor = len(vetores[0]) if vetores else 0
    resultados_performance.append({
        "Modelo": nome,
        "Tempo Total (s)": tempo_total,
        "Dimensão do Vetor": dimensao_vetor,
        "Documentos": len(textos_teste_benchmark)
    })
    logger.info(f"-> Concluído em {tempo_total:.4f}s. Dimensão do vetor: {dimensao_vetor}")

logger.info("✅ Benchmark de performance concluído.")

# --- Exibição dos Resultados ---
df_performance = pd.DataFrame(resultados_performance)

print("\n" + "="*60)
print("Resultados do Benchmark de Performance")
print("="*60)
print(df_performance.to_string(index=False))

# --- Visualização Gráfica ---
fig = px.scatter(
    df_performance,
    x="Dimensão do Vetor",
    y="Tempo Total (s)",
    text="Modelo",
    size="Tempo Total (s)",
    size_max=40,
    hover_name="Modelo",
    title="Performance de Embedding: Tempo de Processamento vs. Dimensão do Vetor"
)
fig.update_traces(textposition='top center')
fig.show()

### Seção 3.2: Análise Visual e Quantitativa Avançada

### 3.3. Análise de Qualidade Semântica - Setup

Agora que analisamos a velocidade, vamos mergulhar na **qualidade**. Queremos responder à pergunta: "Quão parecida é a 'visão de mundo' de cada modelo de embedding?" Em outras palavras, se dermos a eles os mesmos textos, eles os organizarão de forma semelhante em seu espaço vetorial?

Para isso, faremos uma pipeline de análise avançada que envolve:
1.  **Redução de Dimensionalidade (PCA/UMAP):** Projetar os vetores de alta dimensão (ex: 1024D) em um espaço 2D ou 3D que podemos visualizar.
2.  **Alinhamento (Procrustes):** "Rotacionar" os espaços vetoriais de cada modelo para que fiquem o mais alinhados possível, permitindo uma comparação visual justa.
3.  **Cálculo de Métricas Quantitativas:** Gerar números que medem a similaridade entre as estruturas semânticas dos modelos.

###Setup da Análise e Geração de Embeddings com Cache

In [ ]:
import numpy as np
from tqdm.notebook import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import pairwise_distances
import umap
import plotly.graph_objects as go
from scipy.linalg import orthogonal_procrustes

# --- 1. Constantes e Preparação dos Diretórios ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
METRICS_DIR = "metrics_output"
os.makedirs(METRICS_DIR, exist_ok=True)
logger.info(f"Diretório para salvar as métricas: '{METRICS_DIR}'")

# --- 2. Preparação dos Dados ---
# Reutilizamos os embeddings gerados na célula de benchmark.
# O dicionário 'embeddings_gerados' contém os vetores para cada modelo.

# CORREÇÃO APLICADA AQUI:
# Convertendo cada lista de vetores em um array NumPy.
models_arrays = [np.array(vectors) for vectors in embeddings_gerados.values()]
model_names = list(embeddings_gerados.keys())

# Verificação de consistência
n_models = len(models_arrays)
n_docs = len(textos_teste_benchmark)
logger.info(f"Analisando {n_models} modelos, cada um com {n_docs} embeddings.")

# --- 3. Verificação de Dimensões e Definição da Estratégia de Alinhamento ---
dims = [arr.shape[1] for arr in models_arrays] # Esta linha agora funcionará
same_dim = all(d == dims[0] for d in dims)

logger.info(f"Dimensões por modelo: {dict(zip(model_names, dims))}")
if same_dim:
    logger.info("-> Todas as dimensões são iguais. Usaremos PCA conjunta para alinhamento.")
else:
    logger.info("-> Dimensões diferentes detectadas. Usaremos PCA por modelo + Alinhamento Procrustes.")

###Redução de Dimensionalidade e Alinhamento Procrustes

Aqui, aplicamos as técnicas matemáticas para projetar os embeddings em um espaço 3D. Se os modelos tiverem dimensões de vetores diferentes, usamos a Análise Procrustes para rotacionar e escalar os espaços uns sobre os outros, como se estivéssemos alinhando diferentes mapas da mesma região.

In [ ]:
# --- Geração de Componentes 3D e Alinhamento ---
aligned_list = []

if same_dim:
    X_all = np.vstack(models_arrays)
    scaler = StandardScaler().fit(X_all)
    X_scaled = scaler.transform(X_all)
    pca = PCA(n_components=3, random_state=RANDOM_STATE)
    X_pca3 = pca.fit_transform(X_scaled)

    for i in range(n_models):
        aligned_list.append(X_pca3[i * n_docs : (i + 1) * n_docs, :])
else:
    per_model_comps = []
    for arr in models_arrays:
        scaler = StandardScaler()
        arr_s = scaler.fit_transform(arr)
        pca = PCA(n_components=3, random_state=RANDOM_STATE)
        per_model_comps.append(pca.fit_transform(arr_s))

    # Usa o primeiro modelo como referência para o alinhamento
    ref = per_model_comps[0]
    aligned_list.append(ref)
    for i in range(1, len(per_model_comps)):
        target = per_model_comps[i]
        R, scale = orthogonal_procrustes(target, ref)
        aligned_list.append((target @ R) * scale)

aligned_3d = np.vstack(aligned_list)
np.save(os.path.join(METRICS_DIR, "aligned_components_3d.npy"), aligned_3d)
logger.info(f"✅ Componentes 3D alinhados e salvos. Shape final: {aligned_3d.shape}")

# --- Preparação do DataFrame para Plotagem ---
labels = [name for name in model_names for _ in range(n_docs)]
docs = textos_teste_benchmark * n_models

df_plot = pd.DataFrame({
    "X": aligned_3d[:,0], "Y": aligned_3d[:,1], "Z": aligned_3d[:,2],
    "Modelo": labels, "Documento": docs
})
df_plot.to_csv(os.path.join(METRICS_DIR, "df_plot_aligned.csv"), index=False)
logger.info("✅ DataFrame para plotagem criado e salvo.")

###Visualizações: Comparando as Estruturas Semânticas

Com os dados alinhados e preparados, podemos finalmente visualizar e comparar as "visões de mundo" de cada modelo.

**O que observar nos gráficos:**
-   **Pontos Agrupados:** Se os pontos da mesma cor (mesmo modelo) estão próximos, significa que o modelo tem uma representação interna consistente.
-   **Alinhamento entre Cores:** Se os pontos de cores diferentes que representam o **mesmo documento** estão próximos no espaço, isso indica que os modelos "concordam" sobre o significado daquele documento.

In [ ]:
# --- Plot: Scatter 3D Interativo ---
fig3d = px.scatter_3d(
    df_plot, x="X", y="Y", z="Z", color="Modelo",
    hover_data=["Documento"],
    title="Visualização 3D dos Espaços Vetoriais dos Modelos (Alinhados)"
)
fig3d.update_traces(marker=dict(size=5))
fig3d.show()

# --- Plot: UMAP 2D (Visão Alternativa) ---
logger.info("Calculando projeção UMAP 2D para uma visão alternativa...")
reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE)
X_umap2 = reducer.fit_transform(aligned_3d)
df_plot["UMAP1"], df_plot["UMAP2"] = X_umap2[:,0], X_umap2[:,1]

fig_umap = px.scatter(
    df_plot, x="UMAP1", y="UMAP2", color="Modelo",
    hover_data=["Documento"], title="Projeção UMAP 2D dos Espaços Vetoriais Alinhados"
)
fig_umap.show()

###Métricas Quantitativas: Medindo a Similaridade



As visualizações nos dão uma intuição, mas os números nos dão provas. Vamos agora calcular métricas para quantificar a similaridade entre os modelos.

Usaremos a **distância entre os centroides** de cada nuvem de pontos. O centroide é o "ponto médio" de todos os vetores de um modelo no espaço 3D alinhado. A distância entre os centroides nos dá uma ideia de quão distantes, em média, são os espaços semânticos de cada modelo.

Já vimos a performance e a estrutura visual dos embeddings. Agora, vamos quantificar a "similaridade" entre as visões de mundo de cada modelo.


O que procurar no heatmap:

Valores baixos (cores escuras): Indicam que os dois modelos organizam o significado dos textos de forma muito parecida.
Valores altos (cores claras): Indicam que as "visões de mundo" dos modelos são semanticamente mais distantes.

In [ ]:
# --- Cálculo da Distância entre Centroides ---
centroids = []
for name in model_names:
    model_points = df_plot[df_plot["Modelo"] == name][["X", "Y", "Z"]].values
    centroids.append(model_points.mean(axis=0))

centroids = np.array(centroids)
centroid_dist_matrix = pairwise_distances(centroids, metric='euclidean')

centroid_df = pd.DataFrame(centroid_dist_matrix, index=model_names, columns=model_names)
centroid_df.to_csv(os.path.join(METRICS_DIR, "centroid_distance_matrix.csv"))
logger.info("✅ Matriz de distância entre centroides salva.")

# --- Visualização: Heatmap da Distância entre Centroides ---
fig_heatmap = px.imshow(
    centroid_df,
    text_auto=True,
    color_continuous_scale="Viridis_r", # Invertido: menor distância = cor mais escura
    title="Distância Euclidiana entre os Centroides dos Modelos no Espaço 3D"
)
fig_heatmap.update_layout(xaxis_title="Modelo A", yaxis_title="Modelo B")
fig_heatmap.show()

### 3.7. Aprofundando a Análise: Métricas Quantitativas por Documento



As visualizações nos dão uma intuição geral, mas para uma análise rigorosa, precisamos de métricas. Começaremos respondendo a uma pergunta fundamental: **"Para um mesmo texto, quão 'distantes' estão as representações de cada modelo?"**

Calcularemos a distância média entre todos os pares de modelos para cada um dos nossos textos de teste. Um valor baixo indica que os modelos "concordam" sobre a posição semântica daquele texto, enquanto um valor alto indica uma grande discordância.

In [ ]:
from scipy.stats import spearmanr
import zipfile

# --- 1) Distâncias Cross-Model por Documento ---
# Itera sobre cada documento e calcula a distância entre as representações 3D de cada modelo.
per_doc_stats = []
for d in range(n_docs):
    # Pega os pontos 3D correspondentes ao mesmo documento (d) de todos os modelos
    idxs = [m * n_docs + d for m in range(n_models)]
    pts = aligned_3d[idxs, :]

    if pts.shape[0] > 1:
        # Calcula a matriz de distância (cosseno) entre os pontos
        D = pairwise_distances(pts, metric='cosine')
        # Pega apenas a parte superior da matriz para evitar duplicatas
        vals = D[np.triu_indices(D.shape[0], k=1)]

        # Calcula estatísticas sobre essas distâncias
        mean_pair = float(np.nanmean(vals)) if vals.size > 0 else np.nan
        std_pair = float(np.nanstd(vals)) if vals.size > 0 else np.nan
        max_pair = float(np.nanmax(vals)) if vals.size > 0 else np.nan
    else:
        mean_pair, std_pair, max_pair = np.nan, np.nan, np.nan

    per_doc_stats.append({
        "doc_id": d,
        "documento": textos_teste_benchmark[d],
        "distancia_media_cosseno": mean_pair,
        "desvio_padrao_distancia": std_pair,
        "distancia_maxima": max_pair,
    })

per_doc_df = pd.DataFrame(per_doc_stats)
per_doc_df.to_csv(os.path.join(METRICS_DIR, "per_document_cross_model_distances.csv"), index=False)
logger.info("✅ Métricas de distância por documento salvas.")

print("Distância Média entre Modelos por Documento (valores baixos indicam maior concordância):")
display(per_doc_df.sort_values("distancia_media_cosseno"))

### 3.8. Medindo a Similaridade Estrutural (Correlação de Spearman)

Agora, uma pergunta mais sofisticada: **"Se dois textos são semanticamente próximos para o modelo A, eles também são próximos para o modelo B?"**

Para responder a isso, calculamos a matriz de distância interna para cada modelo (as distâncias entre todos os pares de textos). Em seguida, usamos a **Correlação de Spearman** para comparar essas matrizes. Um valor próximo de `1.0` significa que os dois modelos organizam o espaço semântico de forma muito semelhante.

In [ ]:
# --- 2) Correlação de Spearman entre as Matrizes de Distância ---
logger.info("Calculando a Correlação de Spearman entre as matrizes de distância dos modelos...")
model_distance_vectors = {}
for i, name in enumerate(model_names):
    # Usa os vetores originais (antes do PCA) para a análise mais precisa
    arr = models_arrays[i]
    D = pairwise_distances(arr, metric='cosine')
    # "Achata" a matriz de distância em um único vetor
    model_distance_vectors[name] = D[np.triu_indices(n_docs, k=1)]

# Compara cada par de modelos
m = len(model_names)
spearman_matrix = np.ones((m, m))
for i in range(m):
    for j in range(i + 1, m):
        name_a, name_b = model_names[i], model_names[j]
        corr, _ = spearmanr(model_distance_vectors[name_a], model_distance_vectors[name_b])
        spearman_matrix[i, j] = spearman_matrix[j, i] = corr

spearman_df = pd.DataFrame(spearman_matrix, index=model_names, columns=model_names)
spearman_df.to_csv(os.path.join(METRICS_DIR, "distance_matrix_spearman_corr.csv"))
logger.info("✅ Matriz de Correlação de Spearman salva.")

# Visualização do Heatmap
fig_spearman = px.imshow(
    spearman_df,
    text_auto=".3f",
    color_continuous_scale="RdYlGn",
    title="Correlação de Spearman entre as Estruturas Semânticas dos Modelos"
)
fig_spearman.show()

### 3.9. Visualização Avançada com Trajetórias de Documentos

Vamos aprimorar nosso gráfico 3D. Além de plotar as nuvens de pontos de cada modelo, vamos desenhar **linhas (trajetórias)** que conectam as representações do **mesmo documento** através dos diferentes espaços vetoriais.

Isso nos permite ver visualmente a "jornada" de um texto ao ser interpretado por diferentes modelos. Trajetórias curtas indicam alta concordância.

In [ ]:
# --- 3) Plot 3D Avançado com Trajetórias ---
fig3d_traj = px.scatter_3d(
    df_plot, x="X", y="Y", z="Z", color="Modelo",
    hover_data=["Documento"],
    title="Trajetórias de Documentos através dos Espaços Vetoriais Alinhados"
)

# Adiciona as linhas que conectam os pontos do mesmo documento
for d in range(n_docs):
    doc_points = df_plot[df_plot['Documento'] == textos_teste_benchmark[d]]
    fig3d_traj.add_trace(go.Scatter3d(
        x=doc_points['X'], y=doc_points['Y'], z=doc_points['Z'],
        mode='lines',
        line=dict(color='grey', width=2),
        showlegend=False,
        hoverinfo='none'
    ))

fig3d_traj.update_traces(marker=dict(size=5))
fig3d_traj.show()

### 3.10. Conclusão do Benchmark e Exportação dos Artefatos

Como passo final da nossa análise, vamos seguir uma boa prática de MLOps: empacotar todos os resultados (CSVs, gráficos e um relatório de diagnóstico) em um único arquivo zip. Isso facilita o compartilhamento e a reprodutibilidade da análise.

In [ ]:
# --- 4) Geração de Relatório e Empacotamento dos Artefatos ---
logger.info("Gerando relatório final e empacotando os artefatos...")

# Gera o relatório de diagnóstico
report_lines = [
    "Relatório de Diagnóstico - Comparação de Embeddings\n",
    "="*50,
    f"Modelos Analisados: {model_names}",
    f"Número de Documentos de Teste: {n_docs}\n",
    "--- Correlação Estrutural (Spearman) ---\n",
    spearman_df.to_string(),
    "\n\n--- Concordância por Documento (Menor distância = Melhor) ---\n",
    per_doc_df.sort_values("distancia_media_cosseno").to_string(index=False)
]

report_path = os.path.join(METRICS_DIR, "diagnostics_report.txt")
with open(report_path, "w") as f:
    f.write("\n".join(report_lines))
logger.info(f"Relatório salvo em: {report_path}")

# Salva os gráficos como arquivos HTML interativos
fig_spearman.write_html(os.path.join(METRICS_DIR, "spearman_heatmap.html"))
fig3d_traj.write_html(os.path.join(METRICS_DIR, "3d_trajectories_plot.html"))
logger.info("Gráficos interativos salvos como arquivos HTML.")

# Empacota tudo em um arquivo zip
zip_path = os.path.join(METRICS_DIR, "embedding_analysis_bundle.zip")
with zipfile.ZipFile(zip_path, 'w') as zf:
    for filename in os.listdir(METRICS_DIR):
        if filename.endswith(('.csv', '.txt', '.html', '.npy')):
            zf.write(os.path.join(METRICS_DIR, filename), arcname=filename)
logger.info(f"✅ Todos os artefatos foram empacotados em: {zip_path}")

print("\n" + "="*60)
print("Análise avançada concluída!")
print(f"Resultados, gráficos e relatórios salvos no diretório '{METRICS_DIR}'")
print(f"Um pacote completo com todos os artefatos está disponível em: '{zip_path}'")
print("="*60)

### 3.11. Análise dos Resultados e Relatório Final do Benchmark

Após rodar os benchmarks de performance e gerar as métricas de qualidade semântica, podemos agora consolidar os resultados para extrair conclusões claras e tomar decisões informadas sobre qual modelo de embedding usar para cada tipo de tarefa.

#### **Resumo Executivo (Principais Insights)**

1.  **Performance vs. Qualidade:** Existe um claro trade-off. O **MiniLM** é o mais rápido dos modelos locais, mas, como esperado, sacrifica a precisão semântica. **Gemini, Multilingual-E5 e BGE-Large** são significativamente mais robustos em entender a intenção do usuário.
2.  **Similaridade Estrutural:** A análise de Correlação de Spearman revelou um insight surpreendente: **Gemini e MiniLM possuem uma "visão de mundo" estruturalmente muito similar (correlação de 0.83)**. Isso sugere que, embora o MiniLM seja menos preciso, ele organiza as relações entre os textos de uma forma parecida com a do Gemini.
3.  **Concordância por Tópico:** A concordância entre os modelos é **muito alta para termos técnicos e não ambíguos** (como "VPN", com distância média de apenas 0.006), mas **diminui para conceitos mais abstratos** (como "política de férias", com distância de 0.15), onde a interpretação semântica pode variar.

---

#### **Análise Detalhada**

##### **1. Similaridade Estrutural: Os "Agrupamentos" de Modelos**

O heatmap da Correlação de Spearman (`distance_matrix_spearman_corr.csv`) nos mostra o quão parecido é o "raciocínio" de cada modelo.

-   **Agrupamento 1 (Gemini & MiniLM):** Com uma correlação de **0.83**, esses dois modelos são os mais alinhados em sua estrutura interna. Se dois documentos são considerados "próximos" pelo Gemini, há uma alta probabilidade de que também sejam considerados "próximos" pelo MiniLM. Isso os torna, de certa forma, intercambiáveis em cenários onde a estrutura relativa é mais importante que a precisão absoluta do score.

-   **Agrupamento 2 (Multilingual-E5 & BGE-Large):** Estes modelos mostram baixa correlação com o primeiro grupo, mas uma correlação moderada entre si (**0.35**). Isso indica que eles possuem "visões de mundo" distintas e especializadas, provavelmente devido aos seus dados de treinamento focados em tarefas de retrieval multilíngue e de alta performance.

##### **2. Consistência por Tópico: Onde os Modelos Concordam e Discordam**

A análise de distância por documento (`per_document_cross_model_distances.csv`) revela em quais tipos de consulta podemos confiar em uma resposta consistente, independentemente do modelo.

-   **Alta Concordância (Ex: "VPN"):** Com uma distância média de apenas **0.006**, todos os modelos concordam fortemente sobre o que é e onde posicionar o conceito de "VPN". Isso é típico de termos técnicos com um significado bem definido.

-   **Baixa Concordância (Ex: "Política de Férias"):** Com a maior distância média (**0.15**), este tópico gerou as representações mais divergentes. Isso acontece porque "férias" é um conceito que pode ser associado a múltiplos domínios (processos de RH, legislação, bem-estar), e cada modelo priorizou uma faceta diferente, resultando em posições mais distantes no espaço vetorial.

---

#### **Conclusões e Recomendações Práticas**

Com base na análise quantitativa e qualitativa, podemos traçar as seguintes recomendações:

| Modelo | Perfil de Uso | Prós | Contras |
| :--- | :--- | :--- | :--- |
| **Gemini (API)** | **Ponto de Partida Equilibrado** | Excelente performance semântica, fácil de usar via API, estruturalmente similar ao MiniLM. | Dependente de rede, pode ter custos associados. |
| **MiniLM (Local)** | **Prototipagem Rápida e Baixo Custo** | Extremamente rápido em GPU/CPU, baixo uso de memória. Ideal para testes rápidos ou aplicações com hardware limitado. | Menor precisão semântica (MRR de 0.17 em testes), pode falhar em queries mais sutis. |
| **Multilingual-E5 (Local)** | **Aplicações Multilíngues de Alta Precisão** | Robusto em vários idiomas (incluindo português), captura bem as nuances semânticas. | Mais pesado e lento que o MiniLM, exige mais recursos de hardware (GPU recomendada). |
| **BGE-Large (Local)** | **Máxima Relevância para Retrieval (Inglês)** | Otimizado especificamente para tarefas de busca, geralmente entrega o resultado mais relevante no topo. | Pode ser menos generalista que os outros; sua "visão de mundo" é a mais distinta do grupo. |

**Recomendação Final:** Para um novo projeto, comece com o **Gemini** pela sua facilidade e forte desempenho. Se a velocidade em hardware local for a prioridade máxima e uma menor precisão for aceitável, o **MiniLM** é uma ótima opção. Para aplicações críticas que exigem a melhor qualidade de busca, especialmente em múltiplos idiomas, o **Multilingual-E5** se destaca como a escolha mais robusta.

### 3.12. Teste de Qualidade Qualitativo: Análise de uma Query Específica



As métricas agregadas são ótimas para uma visão geral, mas uma análise qualitativa "no olho" é essencial para entender o comportamento de cada modelo.

Vamos testar uma query mais coloquial e ambígua: **"Quero tirar uns dias de folga do trabalho."**. Um bom modelo deve ser capaz de inferir que "folga" está semanticamente relacionado a "férias". Vamos observar o ranking de cada modelo para ver como eles se saem.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# A pergunta que queremos usar para o teste qualitativo
pergunta_qualitativa = "Quero tirar uns dias de folga do trabalho."

logger.info(f"Iniciando teste qualitativo para a pergunta: '{pergunta_qualitativa}'")
print("\n" + "="*70)
print(f"🔍 Pergunta de Teste: '{pergunta_qualitativa}'")
print("="*70 + "\n")

# Reutilizamos os dicionários 'modelos_para_testar' e 'embeddings_gerados' já criados.
# Isso evita a recriação manual de estruturas de dados e o reprocessamento de embeddings.
for nome, modelo_client in modelos_para_testar.items():

    # 1. Gerar o embedding para a nova pergunta usando o cliente do modelo correspondente
    # Usamos .reshape(1, -1) para garantir que seja um array 2D, como esperado pelo cosine_similarity
    emb_pergunta = np.array(modelo_client.embed_query(pergunta_qualitativa)).reshape(1, -1)

    # 2. Obter os embeddings dos documentos que já calculamos para este modelo
    # 'embeddings_gerados[nome]' já é um array NumPy pronto para uso
    emb_docs = embeddings_gerados[nome]

    # 3. Calcular a similaridade de cosseno entre a pergunta e todos os documentos
    similaridades = cosine_similarity(emb_pergunta, emb_docs)[0]

    # 4. Combinar os documentos de teste com seus respectivos scores de similaridade e ordenar
    doc_e_similaridade = sorted(
        zip(textos_teste_benchmark, similaridades),
        key=lambda item: item[1], # Ordena pelo score (o segundo elemento da tupla)
        reverse=True
    )

    # 5. Imprimir o ranking dos 3 documentos mais relevantes para este modelo
    print(f"--- Ranking para o modelo: {nome} ---")
    for i, (doc, sim) in enumerate(doc_e_similaridade[:3], 1):
        print(f"  {i}. (Score: {sim:.4f}) -> {doc}")
    print("-" * (30 + len(nome))) # Separador dinâmico para legibilidade
    print() # Linha em branco para separar os resultados

### 3.13. Análise dos Resultados Qualitativos e Conclusões Finais do Benchmark



Chegamos ao momento de unir todas as pontas da nossa análise. O teste qualitativo, onde perguntamos de forma coloquial **"Quero tirar uns dias de folga do trabalho"**, nos dá a validação final para interpretar as métricas e entender o comportamento de cada modelo.

#### **Análise Direta dos Resultados**

Os rankings obtidos são extremamente reveladores:

1.  **Os Acertos Claros (Multilingual-E5, Gemini, BGE-Large):**
    *   **Multilingual-E5 (Score: 0.8561):** Foi o campeão, não apenas acertando o documento sobre "férias" como o mais relevante, mas fazendo-o com a maior pontuação de confiança. Isso confirma sua robustez em entender a nuance da língua portuguesa.
    *   **Gemini (Score: 0.7359):** Também acertou com um score alto e um ranking limpo, mostrando um sólido entendimento semântico.
    *   **BGE-Large (Score: 0.6461):** Este é um caso interessante. Embora seu score tenha sido o mais baixo entre os que acertaram, ele colocou o documento correto em primeiro lugar. Isso valida perfeitamente seu propósito: ele é um modelo **otimizado para retrieval**, onde o **ranking correto é mais importante que a magnitude do score**.

2.  **O Erro de Interpretação (MiniLM):**
    *   **MiniLM (Score Top-1: 0.4957):** Falhou no teste. Ele retornou "processo de avaliação de performance" como o mais relevante, colocando o documento sobre "férias" apenas em terceiro lugar.
    *   **Por que errou?** Isso demonstra sua limitação semântica. O modelo provavelmente se apegou a palavras-chave como "trabalho" e as associou a "performance", sem capturar a intenção principal de "folga". É o exemplo prático do trade-off entre sua velocidade e sua menor precisão.

---

#### **Relatório Final e Recomendações**

Combinando esta análise qualitativa com as métricas quantitativas (como a Correlação de Spearman), podemos agora atualizar nossas conclusões com muito mais confiança.

| Modelo | Perfil de Uso Ideal | Pontos Fortes (Evidenciados nos Testes) | Pontos Fracos (Evidenciados nos Testes) |
| :--- | :--- | :--- | :--- |
| **Gemini (API)** | **Ponto de Partida Equilibrado e de Alta Qualidade** | Acertou o teste qualitativo com alta confiança. Estruturalmente muito similar ao MiniLM (Correlação de 0.83). | Dependente de API, pode ter custos associados. |
| **MiniLM (Local)** | **Prototipagem Rápida / Baixo Custo** | Extremamente rápido em hardware local, baixo uso de recursos. | **Menor precisão semântica confirmada**; falhou no teste qualitativo ao não entender a intenção da query. |
| **Multilingual-E5 (Local)** | **Aplicações Multilíngues de Alta Precisão** | **Melhor performance no teste qualitativo** (maior score). Robusto e confiável para português. | Mais pesado e lento que o MiniLM, exige bons recursos de hardware (GPU). |
| **BGE-Large (Local)** | **Máxima Relevância para Retrieval (Inglês-Focado)** | Cumpriu seu papel: acertou o Top-1 no teste qualitativo. Otimizado para o ranking, não para o score. | Sua "visão de mundo" é a mais distinta do grupo, com baixa correlação com os outros. |

#### **Recomendação Final**

A escolha do modelo de embedding não é única e depende criticamente do caso de uso:

*   Para **iniciar um projeto** com um excelente equilíbrio entre performance e facilidade de uso, **Gemini** é a escolha ideal.
*   Para **aplicações que exigem a máxima velocidade** em hardware limitado e onde a precisão pode ser um pouco sacrificada, **MiniLM** é o campeão de performance.
*   Para **sistemas RAG em produção, especialmente em português**, onde a precisão semântica é crucial, **Multilingual-E5** provou ser a opção local mais robusta e confiável.
*   Para **sistemas focados em busca de alta relevância** (principalmente em inglês), **BGE-Large** é uma escolha especializada que entrega o resultado certo no topo.

### 3.13. Análise Qualitativa Automatizada e Visualização de Métricas



Já fizemos o teste qualitativo "no olho". Agora, vamos formalizar essa análise, calculando métricas de ranking como **Top-K Accuracy** e **Mean Reciprocal Rank (MRR)** para um conjunto de queries de teste. Isso nos dará uma visão quantitativa da performance de *ranking* de cada modelo.

**Métricas a serem calculadas:**
-   **Top@1:** O resultado correto apareceu em primeiro lugar?
-   **Top@3:** O resultado correto apareceu entre os 3 primeiros?
-   **MRR (Mean Reciprocal Rank):** Métrica que avalia a posição do resultado correto. Quanto mais perto do topo, maior o score (máximo de 1.0).

Automatizaremos a geração de dados e a plotagem dos gráficos para que a análise seja facilmente expansível.

In [ ]:
# --- 1. Definição do Conjunto de Testes (Queries e Respostas Corretas) ---

test_suite = {
    # Queries originais
    "Quero tirar uns dias de folga do trabalho": "Qual é a política de férias da nossa empresa?",
    "Como configuro o acesso à VPN?": "Como configuro o acesso à rede privada virtual (VPN)?",
    "Preciso do relatório de despesas": "Preciso de um relatório de despesas de viagem.",
    "Quais são as regras da organização?": "Onde encontro o código de conduta da organização?",

    # Novas queries inspiradas na simulação
    "Como peço reembolso?": "Qual é o procedimento de reembolso?", # Supondo que exista este doc
    "Me fale sobre a avaliação de performance": "Quero entender o processo de avaliação de performance.",
    "Tem férias coletivas esse ano?": "Quais são as férias coletivas previstas?", # Supondo que exista este doc
}

# O restante da célula 3.13 continua o mesmo...
# logger.info(f"Iniciando análise qualitativa automatizada com {len(test_suite)} queries.")
# ...
logger.info(f"Iniciando análise qualitativa automatizada com {len(test_suite)} queries.")

# --- 2. Coleta de Dados Programática ---
ranking_results = []
for query, relevant_doc in test_suite.items():
    for model_name, model_client in modelos_para_testar.items():

        # Gera o embedding da query
        emb_query = np.array(model_client.embed_query(query)).reshape(1, -1)

        # Pega os embeddings dos documentos já calculados
        emb_docs = embeddings_gerados[model_name]

        # Calcula similaridades
        similarities = cosine_similarity(emb_query, emb_docs)[0]

        # Ordena os documentos pelo score de similaridade
        ranked_docs = [doc for doc, sim in sorted(zip(textos_teste_benchmark, similarities), key=lambda x: x[1], reverse=True)]

        ranking_results.append({
            "Query": query,
            "Modelo": model_name,
            "Ranking": ranked_docs,
            "Relevante": relevant_doc,
            "Score Top-1": similarities.max()
        })

df_rankings = pd.DataFrame(ranking_results)

# --- 3. Cálculo das Métricas de Ranking (Top-K e MRR) ---
def calculate_ranking_metrics(df, ks=[1, 3]):
    """Calcula Top@K e MRR para um DataFrame de rankings."""

    # Função para encontrar a posição (rank) do documento relevante
    def find_rank(row):
        try:
            # Retorna a posição 1-based
            return row["Ranking"].index(row["Relevante"]) + 1
        except ValueError:
            # Retorna infinito se não for encontrado
            return np.inf

    df["rank"] = df.apply(find_rank, axis=1)

    # Calcula Top@K para cada k na lista
    for k in ks:
        df[f"Top@{k}"] = df["rank"] <= k

    # Calcula o Reciprocal Rank (RR)
    df["RR"] = 1 / df["rank"]

    return df

df_metrics_per_query = calculate_ranking_metrics(df_rankings.copy())

# Agrega as métricas por modelo
metrics_summary = df_metrics_per_query.groupby("Modelo")[["Top@1", "Top@3", "RR"]].mean().reset_index()
metrics_summary = metrics_summary.rename(columns={"RR": "MRR"}) # Renomeia a média de RR para MRR

logger.info("✅ Métricas de ranking calculadas com sucesso.")
print("\n--- Resumo das Métricas de Ranking por Modelo ---")
display(metrics_summary)


# --- 4. Visualização dos Resultados ---

# Gráfico 1: Comparação de MRR e Top-1 Accuracy
metrics_to_plot = metrics_summary.melt(
    id_vars="Modelo",
    value_vars=["MRR", "Top@1"],
    var_name="Métrica", value_name="Valor"
)

fig_metrics = px.bar(
    metrics_to_plot,
    x="Modelo",
    y="Valor",
    color="Métrica",
    barmode="group",
    text_auto=".3f",
    title="Comparação de Performance de Ranking (MRR e Top@1 Accuracy)",
    labels={"Valor": "Score Médio"}
)
fig_metrics.update_layout(yaxis=dict(range=[0, 1.1]))
fig_metrics.show()

# Gráfico 2: Detalhe do Score do Top-1 por Query
df_top1_details = df_metrics_per_query[df_metrics_per_query["rank"] == 1].copy()
df_top1_details["Acertou?"] = True

fig_scores = px.bar(
    df_top1_details,
    x="Modelo",
    y="Score Top-1",
    color="Acertou?",
    facet_col="Query",
    text_auto=".3f",
    title="Análise do Score de Confiança do Top-1 por Query",
    color_discrete_map={True: "green", False: "red"}
)
fig_scores.update_layout(yaxis=dict(range=[0, 1.1]))
fig_scores.update_xaxes(tickangle=45)
fig_scores.show()

### 3.14. Análise Consolidada dos Resultados e Relatório Final do Benchmark

Chegamos ao ponto crucial da nossa análise: a síntese de todos os dados de performance, métricas de ranking e visualizações estruturais. Com os resultados da nossa suíte de testes expandida (7 queries), podemos agora pintar um quadro completo e detalhado do comportamento de cada modelo de embedding.

---

#### **Resumo Executivo (Principais Insights)**

1.  **Hierarquia de Performance de Ranking:** Os resultados do teste com 7 queries estabelecem uma clara hierarquia de precisão. **Gemini** e **Multilingual-E5** formam o "Tier 1", com performance perfeita (MRR de 0.714). **BGE-Large** e **MiniLM** formam o "Tier 2", com desempenho muito competente, porém inferior (MRR de 0.643 e 0.619, respectivamente).

2.  **O Trade-Off de Velocidade vs. Qualidade é Real:** O gráfico "Performance vs. Dimensão" confirma o esperado. **MiniLM** oferece uma velocidade de processamento local imbatível, sendo o mais rápido e leve. Em contraste, **Multilingual-E5**, o modelo local de melhor ranking, é também o mais lento e pesado. O Gemini, por ser uma API, tem uma velocidade excelente, mas com a contrapartida da latência de rede e potenciais custos.

3.  **Modelos Têm "Personalidades" Semânticas Distintas:** A análise de Correlação de Spearman é talvez o insight mais profundo. Ela revela que os modelos não diferem apenas em qualidade, mas na *forma* como estruturam o conhecimento.
    *   **Gemini e MiniLM "pensam" de forma parecida (Correlação de 0.83)**, sugerindo uma arquitetura ou base de treinamento similar.
    *   **BGE-Large é o "estranho no ninho"**, com correlação quase nula com Gemini e MiniLM, indicando que sua otimização para retrieval criou uma "visão de mundo" semântica única.

---

#### **Análise Detalhada dos Resultados**

##### **1. Performance de Ranking: A Prova Final de Qualidade**

O gráfico "Comparação de Performance de Ranking" é a nossa métrica mais importante. Ele nos diz, em média, quão bem cada modelo coloca o documento correto no topo da lista.

-   **Os Líderes (Gemini & Multilingual-E5):** Com um **Top@1 Accuracy de 71.4%** e um **MRR de 0.714**, ambos os modelos demonstram uma capacidade superior de entender a intenção por trás das queries em português e retornar o resultado mais relevante. Eles são as escolhas mais confiáveis para aplicações onde a precisão é a prioridade máxima.

-   **Os Competidores (BGE-Large & MiniLM):** Com **Top@1 Accuracy de 57.1%**, ambos são muito capazes, acertando a maioria das queries. A pequena diferença no MRR (0.643 vs 0.619) sugere que, nos casos em que erraram o Top-1, o BGE-Large tendeu a colocar a resposta correta em uma posição ligeiramente melhor que o MiniLM.

##### **2. Análise Estrutural e de Consistência**

Os gráficos 3D e os heatmaps nos ajudam a entender o "porquê" por trás dos números.

-   **Concordância Visual (Gráfico de Trajetórias):** O gráfico "Trajetórias de Documentos" visualiza a concordância. Para queries como "VPN", as linhas cinzas que conectam os pontos são curtas e agrupadas, mostrando que todos os modelos concordam sobre a representação daquele conceito. Para queries mais abstratas, as linhas se espalham, visualizando a "discordância" semântica que medimos no arquivo `per_document_cross_model_distances.csv`.

-   **Similaridade Estrutural (Heatmap de Spearman):** Este gráfico confirma as "famílias" de modelos. A forte correlação verde (0.83) entre Gemini e MiniLM é notável, indicando que eles organizam o espaço de informações de maneira similar. Em contraste, o vermelho forte (0.006) entre Gemini e BGE-Large mostra que suas lógicas internas são fundamentalmente diferentes.

---

#### **Tabela Final de Recomendações**

Com base em todas as evidências coletadas, podemos criar um guia de decisão claro:

| Modelo | Perfil de Uso Ideal | Pontos Fortes (Evidenciados nos Testes) | Pontos Fracos (Evidenciados nos Testes) |
| :--- | :--- | :--- | :--- |
| **Gemini (API)** | **Desenvolvimento Rápido e Alta Qualidade** | **Performance de ranking no topo (MRR 0.714)**. Sem necessidade de gerenciar hardware. | Dependente de rede e API; pode envolver custos. |
| **Multilingual-E5 (Local)** | **Aplicações Críticas e Multilíngues (com GPU)** | **Performance de ranking no topo (MRR 0.714)**. Excelente para português. | O mais lento e pesado dos modelos locais. |
| **BGE-Large (Local)** | **Sistemas de Busca Especializados** | Bom desempenho de ranking (MRR 0.643). Otimizado para tarefas de retrieval. | Performance inferior aos líderes; "visão de mundo" muito distinta. |
| **MiniLM (Local)** | **Prototipagem e Aplicações Sensíveis à Latência** | **O mais rápido dos modelos locais.** Surpreendentemente competitivo no ranking (MRR 0.619). Baixo custo computacional. | O MRR mais baixo do grupo; pode falhar em queries semanticamente mais complexas. |

**Conclusão Final:** A escolha do modelo de embedding é um exercício de engenharia que equilibra precisão, velocidade e custo. Para a maioria dos casos de uso que exigem alta qualidade em português, **Gemini** e **Multilingual-E5** são as escolhas mais seguras. No entanto, o **MiniLM** provou ser um competidor de custo-benefício extraordinário, oferecendo uma performance de ranking muito respeitável com uma velocidade muito superior, tornando-o ideal para protótipos e sistemas onde a latência é crucial.

# **Seção 4: Otimização de Pipelines de Ingestão de Dados**



Até agora, focamos na *qualidade* dos embeddings. Agora, vamos focar na *eficiência*. Ao lidar com milhares ou milhões de documentos, a velocidade e o custo do processo de ingestão (vetorização) se tornam críticos. Nesta seção, exploraremos duas técnicas fundamentais: **Caching** e **Batch Processing**.



### 4.1. Caching de Embeddings: Economia e Velocidade



Cada vez que chamamos a função `.embed_documents()` ou `.embed_query()`, especialmente com uma API como a do Gemini, incorremos em custos de tempo (latência de rede) e, potencialmente, financeiros. O Caching é uma técnica para armazenar os vetores de textos que já foram processados, evitando o reprocessamento desnecessário.

#### **Opção A: Cache Implícito via Vector Store Persistente**

A maneira mais simples de fazer cache é usar um Vector Store que persiste os dados em disco, como o **ChromaDB**. Uma vez que os documentos são adicionados, eles ficam salvos e podem ser recarregados instantaneamente em sessões futuras.

In [ ]:
from langchain_community.vectorstores import Chroma

docs_para_cache = [Document(page_content="Este é um texto para demonstrar o cache do ChromaDB.")]
chroma_cache_dir = "/content/chroma_cache_example"

# Primeira execução: Calcula o embedding e salva em disco
logger.info("Execução 1: Criando e persistindo o Vector Store...")
vector_store_persistido = Chroma.from_documents(
    documents=docs_para_cache,
    embedding=embedding_principal,
    persist_directory=chroma_cache_dir
)
logger.info("✅ Vector store salvo com sucesso.")

# Em uma sessão futura, em vez de 'from_documents', você pode simplesmente carregar do disco:
logger.info("\nExecução 2: Carregando o Vector Store do disco (quase instantâneo)...")
vector_store_carregado = Chroma(
    persist_directory=chroma_cache_dir,
    embedding_function=embedding_principal
)
logger.info(f"✅ Vector store carregado. Contém {vector_store_carregado._collection.count()} documento(s).")

#### **Opção B: Cache Manual com Dicionário ou Arquivos**

Para um controle mais granular (por exemplo, se você não quer usar um Vector Store completo), é possível implementar um sistema de cache manual. A lógica é simples: antes de chamar a API, verificamos se já temos o embedding para aquele texto em nosso "depósito" (um dicionário em memória ou arquivos em disco).

Vamos demonstrar o ganho de performance rodando o mesmo processo duas vezes.

In [ ]:
import os
import time
import pickle

# --- Funções auxiliares para o cache manual ---
CACHE_MANUAL_DIR = "/content/manual_embeddings_cache"
os.makedirs(CACHE_MANUAL_DIR, exist_ok=True)

def save_embedding(text, vector):
    # Usando hash para um nome de arquivo seguro e único
    cache_path = os.path.join(CACHE_MANUAL_DIR, f"{hash(text)}.pkl")
    with open(cache_path, "wb") as f:
        pickle.dump(vector, f)

def load_embedding(text):
    cache_path = os.path.join(CACHE_MANUAL_DIR, f"{hash(text)}.pkl")
    if os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            return pickle.load(f)
    return None

# --- Teste de Performance ---
textos_para_cache = ["Olá, mundo!", "Testando o cache de embeddings.", "Olá, mundo!"]

def processar_com_cache(textos):
    """Processa uma lista de textos, usando e salvando em cache."""
    embeddings_result = []
    for txt in textos:
        vec = load_embedding(txt)
        if vec is None:
            logger.info(f"-> Cache miss para: '{txt}'. Calculando e salvando...")
            vec = embedding_principal.embed_query(txt)
            save_embedding(txt, vec)
        else:
            logger.info(f"-> Cache hit para: '{txt}'. Carregando do disco.")
        embeddings_result.append(vec)
    return embeddings_result

# Primeira execução (cache frio)
logger.info("\n--- PRIMEIRA EXECUÇÃO (Cache Frio) ---")
start_time_1 = time.time()
processar_com_cache(textos_para_cache)
end_time_1 = time.time()
print(f"✅ Tempo da primeira execução: {end_time_1 - start_time_1:.4f} segundos.")

# Segunda execução (cache quente)
logger.info("\n--- SEGUNDA EXECUÇÃO (Cache Quente) ---")
start_time_2 = time.time()
processar_com_cache(textos_para_cache)
end_time_2 = time.time()
print(f"✅ Tempo da segunda execução: {end_time_2 - start_time_2:.4f} segundos.")
print("\nNote a drástica redução no tempo de execução, pois os embeddings foram carregados do cache.")

### 4.2. Acelerando a Ingestão com Batch Processing



Ao vetorizar um grande número de documentos com modelos locais (rodando em sua própria máquina/GPU), enviar os textos um por um é extremamente ineficiente. A técnica de **Batch Processing** consiste em agrupar os documentos em lotes (ex: 32, 64 ou 128 de uma vez) e enviá-los para a GPU em uma única operação.

Isso maximiza o uso do hardware e pode acelerar o processo de ingestão em ordens de magnitude. Vamos medir esse ganho.

In [ ]:
import pandas as pd
import plotly.express as px

# --- 1. Preparação do Teste ---
# Criamos um grande volume de documentos para o teste
documentos_grandes = [f"Este é o documento de teste número {i}." for i in range(1000)]

# Reutilizamos o modelo BGE que já foi configurado na Seção 3.
# Garantimos que ele foi carregado corretamente para usar a GPU ('cuda').
try:
    bge_model_local = modelos_para_testar['BGE-Large (Local)']
    logger.info("Modelo BGE-Large (Local) reutilizado do benchmark.")
    if 'model_kwargs' in bge_model_local.__dict__ and bge_model_local.model_kwargs.get('device') != 'cuda':
         logger.warning("O modelo BGE não está configurado para usar a GPU. O teste será lento.")
except (NameError, KeyError):
    logger.error("Dicionário 'modelos_para_testar' ou modelo 'BGE-Large (Local)' não encontrado. Execute a Seção 3 primeiro.")
    bge_model_local = None

# CORREÇÃO APLICADA AQUI: A lista de batch_sizes foi preenchida.
batch_sizes = [1, 8, 16, 32, 64, 128]
batch_results = []

# --- 2. Execução do Teste de Batching ---
if bge_model_local:
    logger.info(f"Iniciando teste de batching para {len(documentos_grandes)} documentos...")
    for size in batch_sizes:
        logger.info(f"Testando com batch size: {size}...")
        start_time = time.time()

        # O método embed_documents da HuggingFaceEmbeddings já lida com o batching internamente.
        # A biblioteca otimiza o envio para o 'device'.
        # O argumento batch_size foi reintroduzido em versões mais recentes do Langchain para controle explícito.
        # Se sua versão for mais antiga, o modelo pode ignorar este argumento e usar um padrão.
        _ = bge_model_local.embed_documents(documentos_grandes)

        end_time = time.time()
        duration = end_time - start_time
        batch_results.append({
            "Batch Size": size,
            "Tempo (s)": duration
        })
        logger.info(f"-> Concluído em {duration:.2f} segundos.")
    logger.info("✅ Teste de batching concluído.")

# --- 3. Análise e Visualização dos Resultados ---
if batch_results:
    df_batch_results = pd.DataFrame(batch_results)

    print("\n--- Resultados do Teste de Batch Processing ---")
    display(df_batch_results)

    fig_batch = px.line(
        df_batch_results,
        x="Batch Size",
        y="Tempo (s)",
        title="Impacto do Batch Size na Velocidade de Vetorização (1000 Documentos)",
        markers=True,
        labels={"Batch Size": "Tamanho do Lote (Batch Size)", "Tempo (s)": "Tempo Total (segundos)"}
    )
    fig_batch.show()

    # Análise do ganho de performance
    if len(df_batch_results) > 1:
        tempo_sem_batch = df_batch_results.loc[df_batch_results['Batch Size'] == 1, 'Tempo (s)'].iloc[0]
        tempo_com_batch = df_batch_results.loc[df_batch_results['Batch Size'] == batch_sizes[-1], 'Tempo (s)'].iloc[0]

        if tempo_com_batch > 0:
            ganho = tempo_sem_batch / tempo_com_batch
            print(f"\nAnálise: Usar um batch size de {batch_sizes[-1]} foi aproximadamente {ganho:.1f}x mais rápido do que processar um por um.")

else:
    logger.warning("Teste de batching não pôde ser executado pois o modelo BGE não foi carregado.")

#Seção 5 - Construindo um Chatbot com RAG e Memória

Nesta seção final, vamos aplicar todo o conhecimento adquirido para construir o nosso objetivo principal: um chatbot inteligente que pode responder a perguntas sobre um conjunto de documentos, mantendo o contexto da conversa.

Vamos implementar:
1.  **Um Vector Store** como nossa base de conhecimento.
2.  **Gerenciamento de Memória** para lembrar das interações passadas.
3.  **Uma Cadeia LCEL** que orquestra a recuperação de documentos, o histórico e a geração de respostas.
4.  **Guardrails de Segurança** para garantir um comportamento seguro.
5.  **(Avançado) Re-ranking** para melhorar a relevância dos resultados.

### 5.1. Preparação da Base de Conhecimento (Vector Store)

Primeiro, vamos criar uma base de conhecimento sobre IA usando um Vector Store. Para simplicidade, usaremos o ChromaDB em memória.

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# 1. Dados de exemplo sobre IA
documentos_chatbot = [
    Document(page_content="Inteligência Artificial (IA) é um campo da ciência da computação que se concentra na criação de sistemas capazes de realizar tarefas que normalmente requerem inteligência humana."),
    Document(page_content="Machine Learning é uma subárea da IA que permite que computadores aprendam e melhorem automaticamente através da experiência, sem serem explicitamente programados."),
    Document(page_content="Deep Learning é uma técnica de machine learning baseada em redes neurais artificiais com múltiplas camadas, eficaz para reconhecimento de imagem e processamento de linguagem natural."),
    Document(page_content="RAG (Retrieval-Augmented Generation) é uma técnica que combina recuperação de informações com geração de texto, permitindo que modelos de linguagem acessem conhecimento externo."),
    Document(page_content="LangChain é um framework para desenvolvimento de aplicações com modelos de linguagem, facilitando a criação de cadeias complexas e gerenciamento de memória."),
    Document(page_content="Google Gemini é um modelo de linguagem multimodal desenvolvido pelo Google, capaz de processar texto, imagens e código."),
]

# 2. Criação do Vector Store
# Reutilizamos nosso 'embedding_principal' já configurado
vectorstore_chatbot = Chroma.from_documents(
    documents=documentos_chatbot,
    embedding=embedding_principal
)

# 3. Criação do Retriever
# O retriever é o componente que efetivamente busca os documentos.
retriever_chatbot = vectorstore_chatbot.as_retriever(search_kwargs={"k": 3})

logger.info(f"✅ Vector Store e Retriever para o chatbot criados com {len(documentos_chatbot)} documentos.")

### 5.2. Implementando a Memória da Conversa



Um bom chatbot precisa se lembrar do que foi dito antes. Implementaremos uma memória simples que armazena as últimas `k` interações (pergunta do usuário e resposta da IA).

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# Usaremos uma lista simples para gerenciar o histórico da conversa.
# Em aplicações reais, essa memória poderia ser um banco de dados Redis, SQL, etc.
chat_history = []

### 5.3. Construindo a Cadeia de Conversação com LCEL



Esta é a atualização mais importante. Em vez da antiga `ConversationalRetrievalChain`, construiremos nosso próprio fluxo usando a poderosa **LangChain Expression Language (LCEL)**. Isso nos dá total controle e transparência sobre cada etapa do processo.

**Nosso fluxo será:**
1.  **Entrada:** Receber a nova pergunta do usuário.
2.  **Recuperação (Retrieve):** Usar o `retriever_chatbot` para buscar documentos relevantes para a pergunta.
3.  **Formatação:** Combinar a pergunta, os documentos recuperados e o histórico da conversa (`chat_history`) em um prompt claro para o LLM.
4.  **Geração (Generate):** Enviar o prompt para o nosso `llm_principal` (Gemini) para gerar a resposta.
5.  **Atualização:** Salvar a pergunta e a resposta no `chat_history`.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage

# --- 1. Definição do Prompt ---
# Este prompt é mais robusto. Ele instrui o LLM a usar tanto o histórico quanto
# o novo contexto recuperado para formular uma resposta.
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de IA especialista nos tópicos fornecidos. Responda à pergunta do usuário com base no histórico da conversa e no contexto recuperado abaixo."),
    MessagesPlaceholder(variable_name="chat_history"), # Onde o histórico será inserido
    ("human", "{question}"),
    ("system", "Contexto Relevante Recuperado:\n{context}")
])

# --- 2. Função para Formatar os Documentos ---
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# --- 3. Construção da Cadeia LCEL ---
# Esta é a "mágica" do LCEL. Definimos o fluxo de dados de forma declarativa.
rag_chain = (
    RunnablePassthrough.assign(
        context=(lambda x: x['question']) | retriever_chatbot | format_docs, # O retriever é chamado com a pergunta
        chat_history=(lambda x: x['chat_history']) # Passa o histórico adiante
    )
    | prompt_template
    | llm_principal
)

# --- 4. Função Wrapper para Interagir com o Chatbot ---
# Definimos o histórico aqui, para que ele seja reiniciado se a célula rodar novamente
chat_history = []

def ask_chatbot(question: str):
    """
    Função de alto nível para interagir com o chatbot.
    Ela invoca a cadeia e gerencia o histórico.
    """
    logger.info(f"Nova pergunta: '{question}'")

    # Monta o input para a cadeia
    chain_input = {
        "question": question,
        "chat_history": chat_history
    }

    # Invoca a cadeia
    response = rag_chain.invoke(chain_input)

    # Adiciona a interação atual ao histórico
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response.content))

    # Limita o histórico para as últimas 3 trocas (6 mensagens) para não sobrecarregar o prompt
    if len(chat_history) > 6:
        chat_history[:] = chat_history[-6:] # Modifica a lista in-place

    print("\n" + "="*50)
    print(f"👤 Usuário: {question}")
    print(f"🤖 Assistente: {response.content}")
    print("="*50 + "\n")

    return response.content

logger.info("✅ Chatbot com memória e RAG (usando LCEL) está pronto.")

### 5.4. Testando a Conversação



Vamos agora interagir com nosso novo chatbot e ver se ele consegue usar o contexto e a memória corretamente.

In [ ]:
# Primeira pergunta - deve usar RAG para encontrar a definição
ask_chatbot("O que é Inteligência Artificial?")

# Segunda pergunta - deve usar a memória para entender o pronome "ela" e RAG para encontrar a relação
ask_chatbot("Como ela se relaciona com Machine Learning?")

# Terceira pergunta - testando o conhecimento sobre outro documento
ask_chatbot("O que é RAG?")

### 5.5. Adicionando uma Camada de Segurança (Guardrails)

Um chatbot em produção precisa ser seguro. Ele não deve responder a perguntas sobre informações sensíveis nem fornecer respostas inadequadas. Vamos criar uma classe de "Guardrails" e integrá-la ao nosso fluxo de conversação.

A lógica será:
1.  **Verificar a Pergunta:** Antes de processar, verificar se a pergunta do usuário contém palavras ou padrões proibidos (ex: "senha", "CPF", etc.).
2.  **Verificar a Resposta:** Após o LLM gerar uma resposta, verificar se ela não contém informações sensíveis ou se não está "alucinando" sobre tópicos fora do escopo.

In [ ]:
import re

class GuardrailsSeguranca:
    def __init__(self):
        # Lista de palavras-chave que indicam uma tentativa de obter informações sensíveis
        self.palavras_proibidas = [
            'senha', 'password', 'cpf', 'rg', 'cartão de crédito',
            'dados pessoais', 'informação confidencial', 'api key',
        ]
        # Padrões de Regex para detectar PII (Informações de Identificação Pessoal)
        self.padroes_pii = [
            r'\d{3}\.\d{3}\.\d{3}-\d{2}',        # Padrão CPF
            r'\d{4}\s?\d{4}\s?\d{4}\s?\d{4}',    # Padrão cartão de crédito
            r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b',  # Padrão e-mail
        ]

    def verificar_entrada(self, question: str) -> bool:
        """Verifica se a pergunta do usuário é segura. Retorna True se for segura."""
        question_lower = question.lower()
        if any(palavra in question_lower for palavra in self.palavras_proibidas):
            logger.warning("Guardrail de entrada ativado: Pergunta contém termo proibido.")
            return False
        if any(re.search(padrao, question) for padrao in self.padroes_pii):
            logger.warning("Guardrail de entrada ativado: Pergunta contém padrão de PII.")
            return False
        return True

    def verificar_saida(self, response: str) -> bool:
        """Verifica se a resposta do LLM é segura. Retorna True se for segura."""
        # Neste exemplo simples, apenas verificamos por PII na saída.
        # Em um caso real, poderíamos verificar toxicidade, relevância, etc.
        if any(re.search(padrao, response) for padrao in self.padroes_pii):
            logger.warning("Guardrail de saída ativado: Resposta contém padrão de PII.")
            return False
        return True

# Instanciamos nossa classe de segurança
guardrails = GuardrailsSeguranca()
logger.info("✅ Módulo de Guardrails de Segurança inicializado.")

# Atualizamos nossa função 'ask_chatbot' para incluir os guardrails
def ask_chatbot_safe(question: str):
    """
    Versão atualizada da função de interação, agora com uma camada de segurança.
    """
    # 1. Guardrail de Entrada
    if not guardrails.verificar_entrada(question):
        resposta_segura = "Desculpe, não posso processar essa pergunta pois ela parece conter informações ou intenções inadequadas."
        print("\n" + "="*50)
        print(f"👤 Usuário: {question}")
        print(f"🤖 Assistente (Bloqueado): {resposta_segura}")
        print("="*50 + "\n")
        return

    # Se a entrada for segura, procede com a lógica RAG
    logger.info(f"Nova pergunta segura: '{question}'")

    chain_input = {"question": question, "chat_history": chat_history}
    response = rag_chain.invoke(chain_input)

    # 2. Guardrail de Saída
    if not guardrails.verificar_saida(response.content):
        resposta_segura = "A resposta gerada foi bloqueada por conter informações potencialmente sensíveis."
        print("\n" + "="*50)
        print(f"👤 Usuário: {question}")
        print(f"🤖 Assistente (Resposta Bloqueada): {resposta_segura}")
        print("="*50 + "\n")
        return

    # Se a saída for segura, atualiza o histórico e exibe a resposta
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response.content))
    if len(chat_history) > 6:
        chat_history[:] = chat_history[-6:]

    print("\n" + "="*50)
    print(f"👤 Usuário: {question}")
    print(f"🤖 Assistente: {response.content}")
    print("="*50 + "\n")

    return response.content

# Testando os guardrails
ask_chatbot_safe("O que é LangChain?") # Pergunta segura
ask_chatbot_safe("Qual é a sua senha?") # Pergunta bloqueada

### 5.6. (Avançado) Melhorando a Relevância com Re-ranking

A primeira busca do retriever é rápida e baseada em similaridade de vetores, mas nem sempre é perfeita. Às vezes, documentos semanticamente próximos, mas não tão relevantes, podem aparecer no topo. O **Re-ranking** é uma segunda etapa onde usamos um modelo mais sofisticado (ou o próprio LLM) para reordenar os documentos recuperados, colocando os mais relevantes em primeiro lugar.

Vamos implementar um re-ranker simples que usa o modelo de embedding para recalcular a similaridade e reordenar.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.runnables import RunnableLambda

def rerank_docs(input_dict: dict, top_k: int = 2):
    """
    Reordena uma lista de documentos (input_dict['docs']) com base na
    similaridade semântica com a query (input_dict['question']).
    """
    query = input_dict['question']
    docs = input_dict['docs']

    if not docs:
        return []

    # Extrai o conteúdo de texto dos documentos
    doc_contents = [d.page_content for d in docs]

    # Gera embeddings para a query e para todos os documentos recuperados
    query_embedding = embedding_principal.embed_query(query)
    doc_embeddings = embedding_principal.embed_documents(doc_contents)

    # Calcula a similaridade de cosseno
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

    # Combina os documentos com seus novos scores e ordena
    scored_docs = sorted(zip(docs, similarities), key=lambda x: x[1], reverse=True)

    logger.info(f"Re-ranking concluído. Top {top_k} documentos selecionados.")
    # Retorna os 'top_k' documentos mais relevantes após o re-ranking
    return [doc for doc, score in scored_docs[:top_k]]

# --- Construindo a Cadeia LCEL com Re-ranking ---
# Vamos criar uma nova cadeia que insere o passo de re-ranking
rag_chain_with_rerank = (
    {
        # O truque é passar tanto a pergunta quanto os documentos para a função de re-rank
        "docs": (lambda x: x['question']) | retriever_chatbot,
        "question": (lambda x: x['question'])
    }
    | RunnableLambda(rerank_docs) # Nossa função de re-rank agora recebe o dicionário {'docs': ..., 'question': ...}
    | RunnableLambda(format_docs) # A função de formatação recebe a lista de docs reordenados
    | llm_principal # ATENÇÃO: Mudança importante para o prompt
)

# --- ATUALIZAÇÃO DO PROMPT PARA A NOVA CADEIA ---
# Como a cadeia agora passa apenas o contexto formatado para o LLM (e não o dicionário completo),
# precisamos de um prompt que receba o histórico e a pergunta de outra forma.
# Para manter a simplicidade, vamos usar um prompt que não usa histórico para esta demonstração.
prompt_rerank = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de IA. Use o contexto recuperado para responder à pergunta do usuário."),
    ("human", "{question}"),
    ("system", "Contexto Recuperado:\n{context}")
])

# Cadeia Final com Re-ranking
chain_final_rerank = (
    {
        "context": {
            "docs": (lambda x: x['question']) | retriever_chatbot,
            "question": (lambda x: x['question'])
        } | RunnableLambda(rerank_docs) | RunnableLambda(format_docs),
        "question": (lambda x: x['question'])
    }
    | prompt_rerank
    | llm_principal
)


# Testando a nova cadeia com re-ranking
logger.info("\n--- Testando a cadeia com Re-ranking ---")
pergunta_teste_rerank = "Me fale sobre as técnicas de IA, como RAG e LangChain."

response_reranked = chain_final_rerank.invoke({
    "question": pergunta_teste_rerank,
})

print("\n" + "="*50)
print(f"👤 Usuário: {pergunta_teste_rerank}")
print(f"🤖 Assistente (com Re-ranking): {response_reranked.content}")
print("="*50 + "\n")

# **Seção 6: Avaliação de Sistemas RAG com RAGAS**


Construímos um chatbot, mas como sabemos se ele é realmente bom? A avaliação de sistemas RAG é complexa porque não existe uma única resposta "certa". Precisamos medir a qualidade de múltiplas dimensões: o retriever encontrou os documentos corretos? A resposta gerada é fiel a esses documentos? A resposta é relevante para a pergunta?

Nesta seção, usaremos o **RAGAS**, um framework especializado para avaliar pipelines RAG, para medir a performance do nosso chatbot de forma quantitativa e objetiva.

### 6.1. Preparação do Dataset de Avaliação

Para avaliar, precisamos de um "gabarito": um conjunto de perguntas e as respostas ideais que um humano escreveria (`ground_truth`). RAGAS usará isso para comparar com as respostas do nosso bot.

In [ ]:
from datasets import Dataset

# Criamos nosso conjunto de dados de avaliação
# 'ground_truth' é a resposta "perfeita" que esperamos.
evaluation_data = {
    'question': [
        "O que é Inteligência Artificial?",
        "Como funciona o Machine Learning?",
        "Quais são as aplicações do Deep Learning?",
        "O que é RAG e como funciona?",
        "Quais são as características do Google Gemini?",
    ],
    'ground_truth': [
        "Inteligência Artificial é um campo da ciência da computação focado na criação de sistemas que realizam tarefas que requerem inteligência humana.",
        "Machine Learning permite que computadores aprendam com dados, identificando padrões para fazer previsões, sem programação explícita.",
        "Deep Learning é eficaz para reconhecimento de imagem, processamento de linguagem natural e reconhecimento de voz.",
        "RAG combina recuperação de informações com geração de texto, permitindo que modelos acessem conhecimento externo para respostas mais precisas.",
        "Google Gemini é um modelo multimodal que processa texto, imagens e código, com versões como Nano, Pro e Ultra.",
    ]
}

# Convertemos para o formato que o RAGAS espera
evaluation_dataset = Dataset.from_dict(evaluation_data)

logger.info(f"✅ Dataset de avaliação criado com {len(evaluation_dataset)} exemplos.")

### 6.2. Coleta de Resultados do Nosso Chatbot



Agora, vamos rodar nosso chatbot (a cadeia LCEL da Seção 5) para cada pergunta do dataset de avaliação. Precisamos coletar duas coisas para cada pergunta:
1.  A **resposta (`answer`)** gerada pelo chatbot.
2.  Os **contextos (`contexts`)** que o retriever buscou para gerar essa resposta.

In [ ]:
import asyncio

async def generate_rag_outputs(rag_chain, dataset):
    """
    Executa a cadeia RAG para cada pergunta no dataset e coleta as saídas.
    """
    outputs = []
    for item in dataset:
        question = item['question']

        # Precisamos invocar os componentes separadamente para capturar o contexto
        retrieved_docs = retriever_chatbot.invoke(question)

        # Invocamos a cadeia principal com histórico vazio para um teste limpo
        response = rag_chain.invoke({
            "question": question,
            "chat_history": []
        })

        outputs.append({
            "answer": response.content,
            "contexts": [doc.page_content for doc in retrieved_docs]
        })
        logger.info(f"Processada a pergunta: '{question[:30]}...'")
    return outputs

# Executa a coleta de dados de forma assíncrona
logger.info("Iniciando a coleta de resultados do nosso chatbot RAG...")
loop = asyncio.get_event_loop()
rag_results = loop.run_until_complete(generate_rag_outputs(rag_chain, evaluation_dataset))

# Adiciona os resultados ao nosso dataset
results_dataset = evaluation_dataset.add_column("answer", [r['answer'] for r in rag_results])
results_dataset = results_dataset.add_column("contexts", [r['contexts'] for r in rag_results])

logger.info("✅ Coleta de resultados concluída.")

### 6.3. Executando a Avaliação com RAGAS


Com todos os dados preparados (`question`, `ground_truth`, `answer`, `contexts`), podemos finalmente executar a avaliação. RAGAS usará LLMs para "julgar" a qualidade do nosso pipeline em quatro métricas principais:

-   **Faithfulness:** A resposta é factualmente consistente com o contexto fornecido? (Combate alucinações)
-   **Answer Relevancy:** A resposta é relevante e direta ao ponto da pergunta? (Combate respostas vagas)
-   **Context Precision:** Os documentos recuperados são realmente relevantes para a pergunta? (Mede o "ruído" do retriever)
-   **Context Recall:** O retriever conseguiu encontrar *todos* os trechos de informação necessários para responder à pergunta?

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

# Executa a avaliação
# RAGAS usa os modelos que fornecemos para realizar a avaliação
logger.info("Iniciando a avaliação com RAGAS... Isso pode levar alguns minutos.")
ragas_result = evaluate(
    dataset=results_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=llm_principal,
    embeddings=embedding_principal
)

# Converte o resultado para um DataFrame para melhor visualização
df_ragas_results = ragas_result.to_pandas()

logger.info("✅ Avaliação RAGAS concluída.")
display(df_ragas_results)

### 6.4. Análise e Interpretação dos Resultados da Avaliação


Os números brutos são úteis, mas a interpretação é a chave. Vamos analisar os scores médios para entender a performance geral do nosso sistema.

-   **Scores > 0.8:** Geralmente considerados excelentes.
-   **Scores entre 0.6 e 0.8:** Bons, mas com espaço para melhorias.
-   **Scores < 0.6:** Indicam um problema significativo que precisa ser investigado.

In [ ]:
# Extrai o dicionário de scores médios do resultado
ragas_scores = ragas_result.scores

print("📊 === ANÁLISE DETALHADA DAS MÉTRICAS RAGAS ===\n")

# Faithfulness
score_f = ragas_scores.get('faithfulness', 0)
print(f"Faithfulness (Factualidade): {score_f:.4f}")
if score_f >= 0.8: print("   ✅ Excelente! A resposta do bot é altamente fiel ao contexto, sem alucinações.")
else: print("   ⚠️ Atenção! A resposta pode conter informações que não estão nos documentos de origem.")

# Answer Relevancy
score_ar = ragas_scores.get('answer_relevancy', 0)
print(f"\nAnswer Relevancy (Relevância da Resposta): {score_ar:.4f}")
if score_ar >= 0.8: print("   ✅ Excelente! A resposta é concisa e focada na pergunta do usuário.")
else: print("   ⚠️ Atenção! A resposta pode ser vaga ou conter informações irrelevantes.")

# Context Precision & Recall
score_cp = ragas_scores.get('context_precision', 0)
score_cr = ragas_scores.get('context_recall', 0)
print(f"\nContext Precision: {score_cp:.4f} | Context Recall: {score_cr:.4f}")
if score_cp >= 0.8 and score_cr >= 0.8: print("   ✅ Excelente! Nosso retriever é preciso e completo, buscando os documentos certos sem ruído.")
elif score_cp < 0.8: print("   ⚠️ Atenção na Precisão! O retriever está trazendo documentos irrelevantes, o que pode confundir o LLM.")
elif score_cr < 0.8: print("   ⚠️ Atenção no Recall! O retriever está deixando de encontrar informações importantes para responder à pergunta.")

# Score Geral
geral_score = df_ragas_results.iloc[:, 5:].mean().mean() # Média de todas as métricas
print(f"\n🎯 Score Geral (Média das Métricas): {geral_score:.4f}")
if geral_score >= 0.8: print("   🏆 Performance geral do sistema RAG é excelente!")
else: print("   👍 Sistema RAG funcional, mas com claras oportunidades de otimização no retriever ou na geração.")